In [1]:

# ============================================================
# Shared setup (anonymized; paths are never printed)
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr

# Grayscale, publication style; no colour, no titles/captions in saved images.
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"image.cmap": "gray", "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 11, "savefig.dpi": 600})

DATA = Path("../data"); FIG = Path("../results/figures"); TAB = Path("../results/tables")
RNG = np.random.default_rng(42)

# Dataset registry (public sources) and decision domain.
DATASETS = {"telco": "telco_churn.csv", "adult": "adult.parquet",
            "credit_risk": "credit_risk.csv", "gmsc": "gmsc.parquet",
            "taiwan": "taiwan.parquet"}
DOMAIN = {"telco": "non-credit", "adult": "non-credit",
          "credit_risk": "consumer credit", "gmsc": "consumer credit",
          "taiwan": "consumer credit"}
# Grayscale shade + marker per domain (colour-free, print-safe).
STYLE = {"consumer credit": ("0.15", "o"), "corporate credit": ("0.0", "*"),
         "non-credit": ("0.45", "s")}
# Paper-1 anchor (Polish corporate bankruptcy; values reported in the first paper).
POLISH = {"gini": 2*0.778-1, "pbar": 0.0482, "naive_gap": 2.72, "indep_gap": 0.33}

def load(name):
    f = DATA / DATASETS[name]
    return pd.read_parquet(f) if str(f).endswith(".parquet") else pd.read_csv(f)

def prep(name, df):
    # Return numeric design matrix, binary outcome, and two pd-independent drivers (b1,b2).
    df = df.copy()
    if name == "telco":
        y = (df["Churn"].astype(str).str.strip() == "Yes").astype(int).values
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        X = df.drop(columns=["customerID", "Churn"]); b2col = "TotalCharges"
    elif name == "adult":
        y = df["HIGH_INCOME"].astype(int).values
        drop = ["income", "HIGH_INCOME", "AGE_YEARS", "SEX_STD", "EDU_STD", "VIP", "fnlwgt"]
        X = df.drop(columns=[c for c in drop if c in df]); b2col = "hours-per-week"
    elif name == "credit_risk":
        y = df["loan_status"].astype(int).values
        X = df.drop(columns=["loan_status"]); b2col = "person_income"
    elif name == "gmsc":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP"] if c in df]); b2col = "INCOME"
    elif name == "taiwan":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP", "AGE_YEARS", "SEX_STD"] if c in df]); b2col = "LIMIT_BAL"
    b2 = pd.to_numeric(df[b2col], errors="coerce").values if b2col in df else None
    Xn = pd.get_dummies(X, dummy_na=False).apply(pd.to_numeric, errors="coerce")
    return Xn, y, b2

def system_pd(X, y):
    # L2 logistic system model; out-of-fold PD via 5-fold stratified CV; in-fold median imputation.
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    pdh = np.zeros(len(y)); skf = StratifiedKFold(5, shuffle=True, random_state=42)
    for tr, te in skf.split(X, y):
        Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
        med = Xtr.median(numeric_only=True); Xtr = Xtr.fillna(med); Xte = Xte.fillna(med)
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
        clf.fit(sc.transform(Xtr), y[tr]); pdh[te] = clf.predict_proba(sc.transform(Xte))[:, 1]
    return pdh, roc_auc_score(y, pdh)

def pick_b1(X, pdh):
    # Near-orthogonal driver: numeric feature with the lowest |Spearman| vs PD (Attr43 analogue).
    best, br = None, 2.0
    for c in X.columns:
        v = X[c].values.astype(float); m = ~np.isnan(v)
        if m.sum() < 100 or np.nanstd(v) == 0: continue
        r = abs(spearmanr(v[m], pdh[m]).statistic)
        if not np.isnan(r) and r < br: br, best = r, v
    return best

QGRID = [0, 0.10, 0.20, 0.35, 0.55, 0.75, 0.90, 1.0]
def assign_grades(pdh):
    qs = np.quantile(pdh, QGRID); qs[0] -= 1e-9; qs[-1] += 1e-9
    return np.digitize(pdh, qs[1:-1])
SCHED = np.array([0.10, 0.15, 0.20, 0.35, 0.50, 0.70, 0.80])  # ord 0 best .. ord 6 worst
OVR = 0.35

def rank01(v):
    return pd.Series(v).rank(pct=True).fillna(0.5).values

def expected_gap(pi, y):
    # Exact expected upgrade-minus-none default-rate gap (percentage points).
    return (np.sum(pi * y) / np.sum(pi) - y.mean()) * 100.0

def sim_once(pi_firm, grade, G, rng):
    # One generative override draw; returns signed grade_diff (negative = upgrade) and masks.
    N = len(grade)
    overridden = rng.random(N) < OVR
    up = overridden & (rng.random(N) < pi_firm)
    down = overridden & ~up
    notch = np.where(rng.random(N) < 0.70, 1, 2)
    gd = np.zeros(N, int); gd[up] = -notch[up]; gd[down] = notch[down]
    final = np.clip(grade + gd, 0, G - 1)
    return final - grade, overridden, up

def save_fig(fig, name):
    # Save grayscale figure as PNG and PDF at 600 dpi; no caption; no path printed.
    fig.savefig(FIG / (name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(FIG / (name + ".pdf"), bbox_inches="tight")
    plt.close(fig)


# 04_theory_verification.ipynb -- backbone identity, sigma_DR, closed-form C, multi-schedule, general-G
def wmean(x, w): return np.sum(w*x)/np.sum(w)
def wcov(x, y, w):
    mx, my = wmean(x, w), wmean(y, w); return np.sum(w*(x-mx)*(y-my))/np.sum(w)
def wstd(x, w): return np.sqrt(max(wcov(x, x, w), 0))
def grades_G(pdh, G):
    qs = np.quantile(pdh, np.linspace(0, 1, G+1)); qs[0] -= 1e-9; qs[-1] += 1e-9
    return np.digitize(pdh, qs[1:-1])
ANCH = (np.arange(7)+0.5)/7.0
def sched_on(G, s7=SCHED):
    return np.interp((np.arange(G)+0.5)/G, ANCH, s7)
SCHEDS = {"paper": SCHED, "steep": np.array([0.05,0.08,0.12,0.25,0.55,0.80,0.95]),
          "flat": np.array([0.20,0.25,0.30,0.40,0.50,0.60,0.70]),
          "linear": np.linspace(0.10,0.80,7)}
Gs = [7, 14, 28, 70, 200]
ident, sched_tab, G_tab = [], [], []
for name in DATASETS:
    sc = pd.read_parquet(TAB / ("scores_" + name + ".parquet"))
    y = sc["y"].values; pdh = sc["pd"].values; pbar = y.mean()
    gini = 2*__import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(y, pdh) - 1
    U = pd.Series(pdh).rank(pct=True).values
    ident.append({"dataset": name, "cov_emp": round(np.cov(y, U)[0,1], 5),
                  "cov_pred": round(pbar*(1-pbar)*gini/2, 5)})
    for sname, sch in SCHEDS.items():
        g = assign_grades(pdh); w = np.array([(g == k).mean() for k in range(7)])
        DR = np.array([y[g == k].mean() if (g == k).any() else np.nan for k in range(7)])
        gap = expected_gap(sch[g], y); C_emp = (gap/100)/(gini*pbar*(1-pbar))
        rpd = wcov(sch, DR, w)/(wstd(sch, w)*wstd(DR, w))
        Umid = (np.arange(7)+0.5)/7; rmu = wcov(DR, Umid, w)/(wstd(DR, w)*wstd(Umid, w))
        C_pred = np.sqrt(3)*rpd*wstd(sch, w)/(rmu*wmean(sch, w))
        sched_tab.append({"schedule": sname, "dataset": name,
                          "C_emp": round(C_emp, 3), "C_pred": round(C_pred, 3)})
    row = {"dataset": name, "Gini": round(gini, 4)}
    for G in Gs:
        g = grades_G(pdh, G); gap = expected_gap(sched_on(G)[g], y)
        row["C_G%d" % G] = round((gap/100)/(gini*pbar*(1-pbar)), 3)
    G_tab.append(row)
ident = pd.DataFrame(ident); sched_tab = pd.DataFrame(sched_tab); G_tab = pd.DataFrame(G_tab)
ident.to_csv(TAB / "theory_identity.csv", index=False)
sched_tab.to_csv(TAB / "theory_C_by_schedule.csv", index=False)
G_tab.to_csv(TAB / "theory_C_by_G.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ax = axes[0]
ax.plot([0, ident[["cov_emp","cov_pred"]].max().max()*1.1]*1, [0, ident[["cov_emp","cov_pred"]].max().max()*1.1], "k--", lw=1)
ax.scatter(ident["cov_pred"], ident["cov_emp"], color="0.2", s=55)
ax.set_xlabel("predicted  0.5*p(1-p)*Gini"); ax.set_ylabel("empirical  Cov(Y, U)")
ax = axes[1]
greys = {"paper": "0.0", "steep": "0.3", "flat": "0.55", "linear": "0.75"}
for s in SCHEDS:
    sub = sched_tab[sched_tab.schedule == s]
    ax.scatter(sub["C_pred"], sub["C_emp"], color=greys[s], s=45, label=s)
lim = [0.5, 1.6]; ax.plot(lim, lim, "k--", lw=1)
ax.set_xlabel("predicted C (closed form)"); ax.set_ylabel("empirical C"); ax.legend(fontsize=8, title="schedule")
ax = axes[2]
for _, r in G_tab.iterrows():
    ax.plot(Gs, [r["C_G%d" % G] for G in Gs], marker="o", color="0.3", lw=1)
    ax.annotate(r["dataset"], (Gs[-1], r["C_G%d" % Gs[-1]]), textcoords="offset points", xytext=(4, 0), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("number of grades G (log)"); ax.set_ylabel("empirical C")
save_fig(fig, "fig04_theory_verification")
sched_tab.groupby("schedule").agg(C_emp=("C_emp","mean"), C_pred=("C_pred","mean")).round(3)


,C_emp,C_pred
schedule,,
flat,0.636,0.655
linear,0.742,0.764
paper,1.001,1.031
steep,1.391,1.433
